# Harvest Detection

Modified from the DEA Sandbox Vegetation phenology notebook

## Getting started

To run this analysis, run all the cells in the notebook, starting with the "Load packages" cell. 

### Load packages
Load key Python packages and supporting functions for the analysis.

In [3]:
%matplotlib inline

import os
import datacube
import numpy as np
import pandas as pd
import xarray as xr
import datetime as dt
import matplotlib.pyplot as plt
import geopandas as gpd


import sys
sys.path.insert(1, '../Tools/')
from dea_tools.temporal import xr_phenology, temporal_statistics
from dea_tools.datahandling import load_ard
from dea_tools.bandindices import calculate_indices
from dea_tools.plotting import display_map, rgb
from dea_tools.dask import create_local_dask_cluster
from rasterio.features import geometry_mask


import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

## Start Dask Cluster

In [4]:
# Create local dask cluster to improve data load time
client = create_local_dask_cluster(return_client=True) 

/env/lib/python3.10/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 38797 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/e.m.rose@student.uq.edu.au/proxy/38797/status,
Dashboard: /user/e.m.rose@student.uq.edu.au/proxy/38797/status,Workers: 1
Total threads: 96,Total memory: 14.25 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38803,Workers: 0
Dashboard: /user/e.m.rose@student.uq.edu.au/proxy/38797/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:38941,Total threads: 96
Dashboard: /user/e.m.rose@student.uq.edu.au/proxy/36313/status,Memory: 14.25 GiB
Nanny: tcp://127.0.0.1:36195,


## Connect to the datacube

Connect to the datacube so we can access DEA data.
The `app` parameter is a unique name for the analysis which is based on the notebook file name.

In [5]:
dc = datacube.Datacube(app='Harvest_detection')


## Analysis parameters

The following cell sets important parameters for the analysis:

* `veg_proxy`: Band index to use as a proxy for vegetation health e.g. `'NDVI'` or `'EVI'`
* `lat`: The central latitude to analyse (e.g. `-10.6996`).
* `lon`: The central longitude to analyse (e.g. `35.2708`).
* `buffer`: The number of square degrees to load around the central latitude and longitude.
For reasonable loading times, set this as `0.1` or lower.
* `time_range`: The year range to analyse (e.g. `('2019-01', '2019-06')`).



## View the selected location
The next cell will display the selected area on an interactive map.
Feel free to zoom in and out to get a better understanding of the area you'll be analysing.
Clicking on any point of the map will reveal the latitude and longitude coordinates of that point.

## Load cloud-masked Sentinel-2 data

The first step is to load Sentinel-2 data for the specified area of interest and time range. 
The `load_ard` function is used here to load data that has been masked for cloud, shadow and quality filters, making it ready for analysis.

In [6]:
# Load all available Sentinel-2 data lazily
def load_data(query, min_gooddata, num_timesteps, index='NDVI'):
    
    ds = load_ard(
        dc=dc,
        products=[
            'ga_s2am_ard_3',
            'ga_s2bm_ard_3',
            'ga_s2cm_ard_3'
        ],
        cloud_mask='s2cloudless',
        min_gooddata=0.99,
        dask_chunks={'time': 1},
        **query,
    )

    if num_timesteps < len(ds.time):
        # Select evenly distributed observations

        # Exclude November 2019 - was too dry
        ds = ds.sel(
            time=~(
                (ds.time.dt.year == 2019) &
                (ds.time.dt.month == 11)
            )
        )
        
        time_indices = np.linspace(
            0,
            len(ds.time) - 1,
            num_timesteps,
            dtype=int
        )

        ds_sel = ds.isel(time=time_indices)
    else:
        print('Only ' + str(len(ds.time)) + ' timesteps available.')
        ds_sel = ds
        
    
    # Calculate NDVI only for those observations
    ds_sel = calculate_indices(
        ds_sel,
        index=index,
        collection='ga_s2_3'
    )

    ds_sel = ds_sel.compute()

    # filter out any timesteps with NDVI < 0.5 - to avoid any issues with general dryness
    mean_ndvi = ds_sel.NDVI.mean(['x', 'y'])
    
    num_filtered = (mean_ndvi <= 0.5).sum().item()
    if num_filtered > 0:
        filtered_dates = mean_ndvi.time.where(mean_ndvi <= 0.5, drop=True)
        filtered_dates_list = pd.to_datetime(filtered_dates.values).strftime('%Y-%m-%d').tolist()
        print("Filtering out " + str(num_filtered) + " timesteps due to dryness. Dates filtered: ")
        print(filtered_dates_list)
        ds_sel = ds_sel.isel(time=(mean_ndvi > 0.5).values.nonzero()[0])
    
    print("Selected dates:")
    print(ds_sel.time.values)

    print("Data Loaded.")

    return ds_sel

In [7]:
def polygon_harvest(ds, age, threshold=0.20):

    gdf = gpd.read_file("study-site-polygons.shp")
    gdf = gdf.to_crs(ds.odc.geobox.crs)

    transform = ds.odc.geobox.transform
    height = ds.odc.geobox.shape[0]
    width = ds.odc.geobox.shape[1]

    polygon_ndvi = []

    masks = []

    print('Computing mean NDVI...')
    for i, polygon in enumerate(gdf.geometry):
    
        mask = geometry_mask(
            [polygon],
            transform=transform,
            invert=True,
            out_shape=(height, width)
        )
    
        masks.append(mask)
    
        masked = ds[veg_proxy].where(mask)
    
        mean_ndvi = masked.mean(
            dim=['y', 'x'],
            skipna=True
        )
    
        mean_ndvi = mean_ndvi.expand_dims(
            polygon=[i]
        )
    
        polygon_ndvi.append(mean_ndvi)
    
        
    
    polygon_ndvi_xr = xr.concat(
        polygon_ndvi,
        dim='polygon'
    )
        
    
    # Convert to DataFrame
    ndvi_df = polygon_ndvi_xr.to_dataframe().reset_index()

    # First timestep NDVI
    first_ndvi = polygon_ndvi_xr.isel(time=0)
    
    # Minimum NDVI after the first timestep
    min_final_ndvi = (
        polygon_ndvi_xr
        .isel(time=slice(1, None))
        .min(dim='time')
    )
    
    # NDVI drop
    ndvi_drop = first_ndvi - min_final_ndvi

    print('Classifying as harvested/not harvested...')

    gdf['AGE'] = gdf['C_LINKCODE'].map(age.set_index('C_LINKCODE')['AGE'])
    
    gdf['ndvi_diff'] = ndvi_drop.values
    
    gdf['harvested'] = (
        (ndvi_drop >= threshold) &
        (gdf['AGE'] != 0) ## CAN ONLY BE HARVESTED IF NOT FALLOW (AGE = 0)
    ).values
    
    print('Complete.')

    return gdf, ndvi_df

In [8]:
# Set the vegetation proxy to use
veg_proxy = 'NDVI'

# EVI threshold drop needed to be classed as harvested 
threshold = 0.28

# Define area of interest
lat_range = (-29.4336, -29.4841)
lon_range = (153.2438, 153.2906)

# load ages data
ages = pd.read_csv('Results/age_classification.csv')


In [9]:
#create output gpd
gdf = gpd.read_file("study-site-polygons.shp")
harvested = gdf[['C_LINKCODE', 'geometry']]

In [10]:

years = np.arange(2016,2026)

for year in years:
    print('Processing for ' + str(year) + '...')
    query = {
        'x': lon_range,
        'y': lat_range,
        "time": (str(year)+"-06-01", str(year+1)+"-01-31"),
        "resolution": (-10, 10),
        "output_crs": "EPSG:3577",
    }

    age = ages[['C_LINKCODE', str(year)]]
    age = age.rename(columns={str(year): "AGE"})

    ds = load_data(query, min_gooddata=0.99, num_timesteps=12, index=veg_proxy)

    gdf, ndvi_df = polygon_harvest(ds, age, threshold = threshold)

    harvested[str(year)] = harvested['C_LINKCODE'].map(gdf.set_index('C_LINKCODE')['harvested'])

    print()


Processing for 2016...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Counting good quality pixels for each time step using s2cloudless


/env/lib/python3.10/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(
/env/lib/python3.10/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(


Filtering to 11 out of 22 time steps with at least 99.0% good quality pixels
Applying s2cloudless pixel quality/cloud mask
Returning 11 time steps as a dask array
Only 11 timesteps available.
Filtering out 2 timesteps due to dryness. Dates filtered: 
['2016-11-22', '2016-12-02']
Selected dates:
['2016-06-15T23:52:47.458000000' '2016-07-05T23:52:47.461000000'
 '2016-07-25T23:53:14.710000000' '2016-09-23T23:52:40.463000000'
 '2016-10-03T23:52:40.463000000' '2016-10-23T23:52:41.462000000'
 '2016-12-12T23:53:12.931000000' '2017-01-11T23:53:35.795000000'
 '2017-01-31T23:52:38.457000000']
Data Loaded.
Computing mean NDVI...
Classifying as harvested/not harvested...
Complete.

Processing for 2017...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Counting good quality pixels for each time step using s2cloudless
Filtering to 18 out of 41 time steps with at least 99.0% good quality pixels
Applying s2cloudless pixel quality/cloud mask
Returning 18 time steps as a dask arra

In [11]:
harvested_filename = 'Results/harvest_classification.csv'
harvested.to_csv(harvested_filename)
harvested

,C_LINKCODE,geometry,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,1017-00220,"POLYGON ((2028480.223 -3368972.282, 2028472.52...",True,False,False,False,False,True,False,True,False,True
1,1017-00320,"POLYGON ((2028457.36 -3369081.246, 2028456.276...",True,False,True,False,True,False,True,False,True,False
2,1017-00330,"POLYGON ((2028446.244 -3369118.497, 2028443.99...",True,False,True,False,True,False,True,False,True,False
3,1017-00410,"POLYGON ((2028370.568 -3369144.683, 2028323.48...",False,True,False,True,False,True,False,True,True,False
4,1017-00420,"POLYGON ((2028429.797 -3369162.017, 2028421.63...",False,True,False,True,False,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
486,2149-00420,"POLYGON ((2030317.52 -3369566.744, 2030303.958...",False,True,True,False,False,False,False,False,False,False
487,2149-00430,"POLYGON ((2030303.958 -3369609.838, 2030276.10...",False,True,True,False,False,False,False,False,False,False
488,2149-00500,"POLYGON ((2030062.728 -3369445.242, 2030008.15...",True,True,False,False,False,False,False,False,False,False
489,2149-00600,"POLYGON ((2029962.96 -3369412.787, 2029940.047...",False,True,False,False,False,False,False,False,False,False
